# Lesson 12 - 使用代理便签减少聊天历史

本笔记演示如何使用微软代理框架管理长对话中的上下文。随着对话增长，令牌数量增加——最终超过模型的上下文窗口。我们通过**上下文摘要模式**和用于持久记忆的**代理便签**来解决这个问题。

## 你将学到：
1. **为何上下文管理重要**：理解令牌限制和上下文窗口
2. **上下文感知代理**：构建能管理自身对话上下文的代理
3. **上下文摘要模式**：使用工具浓缩对话历史
4. **代理便签**：在上下文压缩后仍能保留的持久记忆

## 前置条件：
- 配置了环境变量的 Azure OpenAI 设置
- 理解前面课程中的基本代理概念


## 设置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## 为什么上下文管理很重要

每个大型语言模型都有一个有限的**上下文窗口**——它在一次请求中可以处理的最大令牌数。随着多轮对话的进行：

- **令牌数量随着每条用户消息和助理回复线性增长**。
- **提示令牌主导成本**，因为每轮都会重新发送整个历史。
- 最终对话**超过上下文窗口**，模型要么截断，要么报错。

### 管理上下文的策略

| 策略 | 工作原理 | 权衡 |
|---|---|---|
| **截断** | 删除最旧的消息 | 失去早期上下文 |
| **摘要** | 将较旧的消息浓缩成摘要 | 细节有所损失，但保留关键点 |
| **草稿板 / 外部记忆** | 将关键事实存储在对话之外 | 需要调用工具，但能承受任意缩减 |

在本笔记本中，我们结合了**摘要**和**草稿板工具**，让代理即使在对话历史被浓缩时也能保持连续性。


## 创建一个上下文感知代理


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## 模拟一场长对话

让我们通过一个多轮对话来看看上下文如何积累。代理应在多轮中保留关键细节（偏好、预算、旅行日期），并展示连续性。


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

请注意代理如何保留之前对话的上下文——它了解日本、寿司、寺庙、摄影、3000美元预算、单独旅行以及四月的时间范围。在简短的对话中这效果很好，但随着对话的继续，完整的历史记录重传变得代价高昂。

让我们继续对话，增加更多轮次，以观察上下文的累积：


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## 上下文总结模式

随着对话的增长，我们可以使用**总结工具**将累计的上下文压缩成紧凑的格式。代理调用此工具来记录关键偏好，即使旧消息被丢弃，重要信息仍能被保存。

该模式是更复杂历史缩减的构建块：
1. 代理识别对话中的关键事实
2. 它调用总结工具来保存这些内容
3. 旧消息可以安全移除，因为总结捕获了重要内容

下面我们定义一个 `summarize_preferences` 工具，代理可以调用它来记录所学内容的紧凑摘要。


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## 总结

在本课中，您学习了如何使用 Microsoft Agent Framework 管理长时间运行的代理对话中的上下文：

### 关键概念
- **上下文窗口是有限的** —— 对话历史中的每个标记都会产生费用并计入限制。
- **摘要工具** 让代理能够将积累的上下文压缩成紧凑的摘要，减少标记使用量，同时保留关键信息。
- **代理便签** 提供持久的外部记忆，能够在任何对话削减后依然保留信息。

### 您构建的内容
- 一个 **上下文感知代理**，能够在多轮对话中保持连续性
- 一个 **摘要工具** (`summarize_preferences`)，以紧凑格式记录关键用户详情
- 一个 **多轮对话** 演示了上下文保留和变化处理

### 现实世界应用
- **客户服务机器人**：在长时间支持会话中记住用户偏好
- **个人助理**：追踪进行中的项目，无需重复解释上下文
- **教育导师**：在多次交互中维护学生进度

### 下一步
- 实现带有基于文件持久化的完整便签工具
- 在摘要后添加自动历史截断
- 与向量数据库结合，实现语义记忆搜索
- 构建能够在数日后带完整上下文恢复对话的代理


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译。尽管我们力求准确，但请注意，自动翻译可能存在错误或不准确之处。原始文档的原文版本应被视为权威来源。对于关键信息，建议采用专业人工翻译。因使用本翻译内容而引起的任何误解或误释，我们概不负责。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
